# Kestrel feedback: source investigation

**Purpose:** understand how the source files differ before building the clean feedback dataset.

Run from top to bottom. Each section asks one question and shows the evidence. This notebook does **not** clean, deduplicate or change source files. Row counts below describe physical worksheet/CSV records, not validated customer responses.

**Presentation route:** file inventory → reading differences → workbook layouts → structural inventory → special cases → decisions for the pipeline.

Source data and notebook previews remain local. Previews mask common identifiers, but masking is heuristic and is not a guarantee that all personal information has been removed. Do not publish raw files or unrestricted notebook outputs.

## 1. Setup — where is the data?

Use `pathlib` so the notebook works from either the assignment folder or its `scripts` folder. No old audit scripts are imported: importing them would execute their top-level code and create unrelated files.

In [ ]:
from pathlib import Path
import csv
import hashlib
import io
import json
import re
from collections import Counter
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((p for p in candidates if (p / "feedback").is_dir() and (p / "reference").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the assignment folder or its scripts subfolder.")
FEEDBACK_DIR = PROJECT_DIR / "feedback"
OUTPUT_DIR = PROJECT_DIR / "scripts" / "investigation_outputs"
print("Assignment folder:", PROJECT_DIR)
print("Feedback folder exists:", FEEDBACK_DIR.is_dir())
print("pandas version:", pd.__version__)

## 2. Inventory — what files did we receive?

List every supported feedback file, its size and SHA-256 hash. Hashes identify file content; filenames alone do not establish whether a file is a new version. Excel temporary lock files are excluded explicitly.

In [ ]:
files = sorted(p for p in FEEDBACK_DIR.iterdir() if p.suffix.lower() in {".csv", ".xlsx"} and not p.name.startswith("~$"))
file_inventory = pd.DataFrame([
    {"filename": p.name, "format": p.suffix.lower(), "bytes": p.stat().st_size,
     "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
    for p in files
])
assert not file_inventory.empty, "No supported files found."
display(file_inventory.groupby("format").agg(files=("filename", "size"), bytes=("bytes", "sum")))
print("Total files:", len(files), "— pack description expects 52; investigate any difference.")
display(file_inventory[["filename", "format", "bytes"]])

## 3. CSV loader — why does a default read fail?

A delimiter controls how fields are separated; an encoding controls how bytes become characters. Decode strictly using UTF-8 with BOM support, then Windows-1252. Do not silently replace broken characters or skip malformed records. Detect comma/semicolon/tab delimiters from the header, and record which settings worked.

The loader preserves strings, empty fields, preamble rows and header rows. It returns both the raw grid and parser metadata. It deliberately refuses ragged records so they can be investigated.

In [ ]:
def load_feedback_csv(filepath):
    filepath = Path(filepath)
    content = filepath.read_bytes()
    for encoding in ("utf-8-sig", "cp1252"):
        try:
            text = content.decode(encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"Unsupported encoding: {filepath.name}")
    first_line = next((line for line in text.splitlines() if line.strip()), "")
    if not first_line:
        raise ValueError(f"Empty CSV: {filepath.name}")
    delimiter = csv.Sniffer().sniff(first_line, delimiters=",;\t").delimiter
    rows = list(csv.reader(io.StringIO(text), delimiter=delimiter, strict=True))
    widths = {len(row) for row in rows if row}
    if len(widths) != 1:
        raise ValueError(f"Ragged CSV records in {filepath.name}: widths={sorted(widths)}")
    grid = pd.DataFrame(rows).fillna("")
    return grid, {"encoding": encoding, "delimiter": repr(delimiter)}

print("CSV loader defined. No files have been changed.")

### Preview helper — keep obvious identifiers out of displayed examples

Hide columns labelled as customer, name, contact, phone, email or registration fields. Also mask common emails, Australian mobile numbers and explicitly labelled registrations in text. Names embedded in prose can remain: this helper is for local investigation, **not** the final AI/export privacy gate.

In [ ]:
def mask_text(value):
    text = str(value) if value is not None else ""
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", "[EMAIL]", text)
    text = re.sub(r"(?<!\d)(?:\+61[ -]?4|04)(?:[ -]?\d){8}(?!\d)", "[PHONE]", text)
    text = re.sub(r"(?i)\b(rego|registration)\s*[:#-]?\s*[A-Z0-9 -]{5,8}\b", "[REGISTRATION]", text)
    return text

def safe_preview(grid, header_row=0, rows=5):
    view = grid.iloc[:rows].copy().astype(object)
    if len(grid) > header_row:
        headers = grid.iloc[header_row].astype(str)
        for col, label in headers.items():
            identity_label = re.search(r"(?i)(name|customer|cust no|customer no|dms_customer|phone|mobile|email|rego|registration|reviewer)", label)
            feedback_label = re.search(r"(?i)(comment|said|feedback|verbatim|review$)", label)
            if identity_label and not feedback_label:
                view.loc[:, col] = "[IDENTIFIER COLUMN]"
    return view.map(mask_text)

print("Local preview masking helper ready.")

### Demonstrate the loader on three different CSV layouts

Inspect the settings and the first few rows. These examples cover a normal comma-separated export, a semicolon-separated export and a Windows-encoded export. CSV row 1 remains visible as data so we do not confuse reading with interpretation.

In [ ]:
for name in ("boxhill_survey_export_2025Q4.csv", "chatswood_2025_q4.csv", "Wollongong_Oct25-Mar26.csv"):
    grid, settings = load_feedback_csv(FEEDBACK_DIR / name)
    print(name, settings, "physical records:", len(grid))
    display(safe_preview(grid))

## 4. Workbooks — inspect every sheet, not just the first

Read without assigning headers and without inferring dates. Retain the raw grids in memory for later inspection. Excel may still decode stored date cells into date objects; this is not a lossless Bronze ingestion. The future ingestion must preserve source bytes, cell types and formula policy separately.

Read failures are recorded, never hidden. Empty sheets and summary/review tabs remain in the inventory.

In [ ]:
raw_grids = {}
parser_settings = {}
read_errors = []
for path in files:
    try:
        if path.suffix.lower() == ".csv":
            grid, settings = load_feedback_csv(path)
            raw_grids[(path.name, "csv")] = grid
            parser_settings[path.name] = settings
        else:
            with pd.ExcelFile(path, engine="openpyxl") as book:
                for sheet in book.sheet_names:
                    try:
                        raw_grids[(path.name, sheet)] = pd.read_excel(book, sheet_name=sheet, header=None, dtype=object, keep_default_na=False)
                    except Exception as exc:
                        read_errors.append({"filename": path.name, "sheet": sheet, "error": str(exc)})
    except Exception as exc:
        read_errors.append({"filename": path.name, "sheet": "", "error": str(exc)})

sheet_overview = pd.DataFrame([
    {"filename": name, "sheet": sheet, "physical_rows": len(grid), "physical_columns": len(grid.columns)}
    for (name, sheet), grid in raw_grids.items()
])
print("Loaded file/sheet pairs:", len(sheet_overview), "Read failures:", len(read_errors))
display(sheet_overview)
if read_errors:
    display(pd.DataFrame(read_errors))

### Inspect the opening rows of a workbook with title rows

Dandenong illustrates why `header=0` is unsafe. Display row positions as one-based spreadsheet row numbers. Change the file and sheet below to inspect another source.

In [ ]:
example_key = ("Dandenong Customer Feedback 2025-26.xlsx", "Oct 25")
example = raw_grids[example_key]
print("File and sheet:", example_key)
preview = safe_preview(example, header_row=3, rows=7)
preview.index = preview.index + 1
preview.index.name = "source_row"
display(preview)

## 5. Structural inventory — where might the headers be?

Suggest header rows only when a date label and a department or feedback label occur together. Scan the **whole** sheet to expose repeated headers and mid-sheet schema changes. Human review is still needed: unmatched sheets may be summaries, notes or a layout the heuristic missed.

Column order and exact spelling are preserved; a second normalized signature helps compare casing and whitespace. No source values are changed.

In [ ]:
DATE_LABELS = {"date", "date received", "submitted_at", "month", "when"}
CONTEXT_LABELS = {"dept", "department", "area", "comments", "comment", "feedback", "verbatim", "review"}
def header_candidates(grid):
    result = []
    for index, row in grid.iterrows():
        values = [str(v).strip().lower() for v in row]
        labels = set(values)
        if labels & DATE_LABELS and labels & CONTEXT_LABELS:
            exact = [str(v) for v in row if str(v).strip()]
            result.append({"source_row": int(index) + 1, "exact_headers": exact,
                           "normalized_headers": [v.strip().lower() for v in exact]})
    return result

structure_records = []
for (name, sheet), grid in raw_grids.items():
    headers = header_candidates(grid)
    structure_records.append({
        "filename": name, "sheet": sheet, "physical_rows": len(grid),
        "physical_columns": len(grid.columns), "header_candidates": headers,
        "candidate_rows": [h["source_row"] for h in headers],
        "header_status": "review: multiple headers" if len(headers) > 1 else "candidate found" if headers else "review: no header candidate"
    })
audit_df = pd.DataFrame(structure_records)
display(audit_df.drop(columns="header_candidates"))
print("Sheets requiring header review:")
display(audit_df[audit_df["header_status"] != "candidate found"].drop(columns="header_candidates"))

### Compare recurring column layouts

A signature identifies an ordered list of normalized column labels. Different signatures are evidence for source mappings, not proof that two layouts have different business meanings. Multiple candidates per sheet appear separately.

In [ ]:
header_records = [
    {"filename": r["filename"], "sheet": r["sheet"], "source_row": h["source_row"],
     "signature": " | ".join(h["normalized_headers"])}
    for r in structure_records for h in r["header_candidates"]
]
headers_df = pd.DataFrame(header_records)
display(headers_df.groupby("signature", sort=True).size().rename("candidate_occurrences").reset_index())
print("Distinct normalized labels:")
print(sorted({label for r in structure_records for h in r["header_candidates"] for label in h["normalized_headers"]}))

## 6. Special investigations — evidence that changes the cleaning rules

Keep each question separate. These examples are observations; no correction, deletion or version precedence is applied here.

### A. Are all numeric ratings on the same scale?

Geelong documents opposite scale directions in its two annual sheets. Ringwood changes layout and score scale within its first-quarter sheet. A single file-level rating rule would be unsafe.

In [ ]:
for sheet in ("2025", "2026"):
    grid = raw_grids[("Geelong Feedback.xlsx", sheet)]
    print("Geelong", sheet)
    display(safe_preview(grid, header_row=3, rows=4))
ringwood_key = ("Ringwood feedback Q1 2026.xlsx", "Feedback")
for header in header_candidates(raw_grids[ringwood_key]):
    print("Ringwood header at source row", header["source_row"], ":", header["exact_headers"])

### B. Are apparent duplicate tabs actually identical?

Compare Penrith's June and July grids as loaded. Equality supports investigating a copied tab; it does **not** prove which month is correct or recover missing June responses. Check the dates and retain both source locations for the later duplicate decision.

In [ ]:
penrith_june = raw_grids[("Penrith Feedback.xlsx", "Jun 26")]
penrith_july = raw_grids[("Penrith Feedback.xlsx", "Jul 26")]
print("June and July grids identical:", penrith_june.equals(penrith_july))
for label, grid in (("Jun 26", penrith_june), ("Jul 26", penrith_july)):
    print(label, "first recorded date:", grid.iloc[1, 0], "physical rows:", len(grid))

### C. What changed between Chermside's FINAL versions?

Compare full row values with multiplicity, rather than counting only unique rows. This detects additions/removals but does not interpret changed records as new customers. Stable references and conflicting versions need an explicit pipeline policy.

In [ ]:
old_key = ("Chermside Q1 2026 FINAL.xlsx", "Feedback")
new_key = ("Chermside Q1 2026 FINAL v2.xlsx", "Feedback")
def row_multiset(grid):
    return Counter(tuple(str(v) for v in row) for row in grid.iloc[1:].itertuples(index=False, name=None))
old_rows, new_rows = row_multiset(raw_grids[old_key]), row_multiset(raw_grids[new_key])
print("Earlier version data rows:", sum(old_rows.values()))
print("v2 data rows:", sum(new_rows.values()))
print("Rows shared, including multiplicity:", sum((old_rows & new_rows).values()))
print("Rows only in earlier version:", sum((old_rows - new_rows).values()))
print("Rows only in v2:", sum((new_rows - old_rows).values()))

### D. Does every tab contain individual feedback?

Toowoomba's summary has rating counts, not one response per row. Southport's copied reviews use relative dates. Keep aggregate and review sources distinguishable; do not invent individual records or exact dates to make them fit a common fact table.

In [ ]:
for key, header_row in [
    (("Toowoomba Q4 2025 summary.xlsx", "Summary"), 2),
    (("Southport Feedback Tracker.xlsx", "Google reviews (copied)"), 1),
]:
    print(key)
    display(safe_preview(raw_grids[key], header_row=header_row, rows=5))

### E. What could be mistaken for missing data?

Report physically blank cells without interpreting them as missing customer answers. Title rows, merged cells and notes inflate these figures. A semantic completeness profile must follow header mapping and removal/classification of non-response rows.

In [ ]:
blank_profile = []
for (name, sheet), grid in raw_grids.items():
    blank = grid.map(lambda v: str(v).strip() == "")
    blank_profile.append({"filename": name, "sheet": sheet, "blank_cells": int(blank.sum().sum()),
                          "all_blank_rows": int(blank.all(axis=1).sum()) if len(grid.columns) else 0})
display(pd.DataFrame(blank_profile))

## 7. Findings → proposed decisions

- **Geelong:** sheet-specific scale direction; preserve the raw score and the normalization rule.
- **Ringwood:** split mapping by header segment, not just by file or sheet.
- **Penrith:** investigate copied June/July content; do not shift dates or count both automatically.
- **Chermside:** resolve overlapping versions using references and explicit precedence; preserve superseded lineage.
- **Toowoomba:** retain aggregate counts separately from individual feedback.
- **Southport reviews:** retain as a separate source; relative dates are not exact event dates.
- **CSV parsing:** record delimiter and encoding; reject malformed input explicitly.
- **Privacy:** source identifiers stay in restricted Bronze; the production masking step needs tests and human review before AI or export.

**Next pipeline work:** reviewed YAML mappings; canonical dealer/date/department/rating fields; row dispositions and reconciliation; DuckDB Bronze/Silver/Gold; tests for these observed traps. AI themes and Tableau follow once the baseline is trustworthy. This notebook is an investigation, not the completed ingestion pipeline.

## 8. Verification and reproducible inspection outputs

Check that every supported file is accounted for, failures are explicit, and the loader preserved known CSV differences. Save structural summaries without raw comments or customer identifiers. Existing files with these output names are refreshed on rerun. Original feedback files are never written.

In [ ]:
accounted_files = {name for name, _ in raw_grids} | {e["filename"] for e in read_errors}
assert accounted_files == {p.name for p in files}, "Some files were not accounted for."
assert len(raw_grids) == len(sheet_overview), "Duplicate file/sheet inventory keys."
assert not read_errors, f"Resolve {len(read_errors)} read failure(s) before trusting the inventory."
assert parser_settings["chatswood_2025_q4.csv"]["delimiter"] == repr(";")
assert parser_settings["Wollongong_Oct25-Mar26.csv"]["encoding"] == "cp1252"
assert len(header_candidates(raw_grids[ringwood_key])) >= 2

OUTPUT_DIR.mkdir(exist_ok=True)
file_inventory.to_csv(OUTPUT_DIR / "file_inventory.csv", index=False)
sheet_overview.to_csv(OUTPUT_DIR / "sheet_inventory.csv", index=False)
headers_df.to_csv(OUTPUT_DIR / "header_candidates.csv", index=False)
(OUTPUT_DIR / "structure_profile.json").write_text(json.dumps(structure_records, indent=2), encoding="utf-8")
(OUTPUT_DIR / "csv_parser_settings.json").write_text(json.dumps(parser_settings, indent=2), encoding="utf-8")
print("Checks passed.")
print("Files:", len(files), "File/sheet pairs:", len(raw_grids), "Read failures:", len(read_errors))
print("Structural outputs saved to:", OUTPUT_DIR)
print("No customer-response counts or business KPIs have been claimed at this stage.")